In [ ]:
import pandas as pd
import numpy as np
import boto3
import io

s3_client = boto3.client('s3')

# Helper function stays the same
def compute_distance(lat1, lon1, lat2, lon2):
    R = 6371.0
    lat1, lon1, lat2, lon2 = map(np.radians, [lat1, lon1, lat2, lon2])
    dlat = lat2 - lat1
    dlon = lon2 - lon1
    a = np.sin(dlat / 2)**2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon / 2)**2
    c = 2 * np.arctan2(np.sqrt(a), np.sqrt(1 - a))
    return R * c

bucket_name = 'credit-card-fd'
file_key = "Notebook/data/combined_result.xlsx"

response = s3_client.get_object(Bucket=bucket_name, Key=file_key)
df = pd.read_excel(io.BytesIO(response['Body'].read()), engine='openpyxl')

# Preview your data to ensure it loaded correctly
df.head()

# Pre-processing: Ensure timestamp is datetime and sorted
df['timestamp'] = pd.to_datetime(df['timestamp'])
df = df.sort_values(['user_id', 'timestamp'])

# --- Feature 1: Amount Deviation ---
df["amount_deviation"] = df["amount"] - df.groupby("user_id")["amount"].transform("mean")

# --- Feature 2: Transaction Velocity (Rolling 3) ---
df["txn_velocity_1h"] = df.groupby("user_id")["amount"].rolling(window=3).sum().reset_index(0, drop=True).fillna(0)

# --- Feature 3: Geo Distance ---
# Map countries to coordinates (Example mapping)
country_coords = {
    'India': (20.5937, 78.9629),
    'United States': (37.0902, -95.7129),
    'Australia': (-25.2744, 133.7751)
}
df['lat'] = df['country_name'].map(lambda x: country_coords.get(x, (0,0))[0])
df['lon'] = df['country_name'].map(lambda x: country_coords.get(x, (0,0))[1])

# Get previous coordinates
df["prev_lat"] = df.groupby("user_id")["lat"].shift(1).fillna(df["lat"])
df["prev_lon"] = df.groupby("user_id")["lon"].shift(1).fillna(df["lon"])

# Compute distance
df["geo_distance"] = compute_distance(df["prev_lat"], df["prev_lon"], df["lat"], df["lon"])

# Cleanup helper columns
df_output = df.drop(columns=['lat', 'lon', 'prev_lat', 'prev_lon'])

csv_buffer = io.BytesIO()
df.to_excel(csv_buffer, index=False, engine='openpyxl')

s3_client.put_object(Bucket='credit-card-fd', Key='Notebook/data/bank_train_data_feature.xlsx', Body=csv_buffer.getvalue())
print("Data successfully processed and saved to S3.")
